In [1]:
import re, unicodedata
import numpy as np
import pandas as pd

df = pd.read_csv('gari_import.csv', dtype=str)
print(f'Loaded: {len(df)} rows')
df.head()

Loaded: 994 rows


,Company,Model,Year,Color,Engine_Capacity,Fuel_Type,Mileage,Auction_Grade,Price
0,TOYOTA,VOXY HYBRID,2020,BLACK,1800,HYBRID,69777,4.5,3500000
1,TOYOTA,ESQUIRE,2020,BLACK,1800,HYBRID,92500,3.5,3700000
2,TOYOTA,ESQUIRE,2020,PEARL,1800,HYBRID,97000,3.5,4000000
3,TOYOTA,COROLLA,2020,PEARL,1800,HYBRID,33000,4.5,3600000
4,Toyota,Corolla Cross,2021,Redwine,1800,Hybrid,30000,4.5,4850000


In [2]:
def norm(x):
    if pd.isna(x): return np.nan
    x = unicodedata.normalize('NFKC', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return np.nan if x == '' or x.lower() in ('unknown', 'nan') else x

df = df.apply(lambda s: s.map(norm))
df

,Company,Model,Year,Color,Engine_Capacity,Fuel_Type,Mileage,Auction_Grade,Price
0,TOYOTA,VOXY HYBRID,2020,BLACK,1800,HYBRID,69777,4.5,3500000
1,TOYOTA,ESQUIRE,2020,BLACK,1800,HYBRID,92500,3.5,3700000
2,TOYOTA,ESQUIRE,2020,PEARL,1800,HYBRID,97000,3.5,4000000
3,TOYOTA,COROLLA,2020,PEARL,1800,HYBRID,33000,4.5,3600000
4,Toyota,Corolla Cross,2021,Redwine,1800,Hybrid,30000,4.5,4850000
...,...,...,...,...,...,...,...,...,...
989,TOYOTA,AXIO HYBRID,2015,SILVER,NaN,52940,NaN,"1,600,000 Taka",NaN
990,TOYOTA,AXIO HYBRID,2015,BEIGE,1500,HYBRID,97000,4,1530000
991,TOYOTA,AXIO HYBRID,2015,SILVER,1500,Gasoline,74000,4,1650000
992,Toyota,CHR,2017,NaN,1800,Hybrid,1880,5,3150000


In [3]:
isnum = lambda v: pd.notna(v) and re.fullmatch(r'[\d,\.]+', str(v)) is not None
taka  = df['Auction_Grade'].str.contains('taka', case=False, na=False)

for i in df.index[taka]:
    df.at[i, 'Price'] = re.sub(r'[^\d]', '', df.at[i, 'Auction_Grade'])
    if isnum(df.at[i, 'Fuel_Type']):
        df.at[i, 'Mileage']        = df.at[i, 'Fuel_Type']
        df.at[i, 'Auction_Grade']  = df.at[i, 'Mileage'] if isnum(df.at[i, 'Mileage']) else np.nan
        df.at[i, 'Fuel_Type']      = np.nan
    else:
        df.at[i, 'Auction_Grade']  = np.nan

print(f'Shifted rows repaired: {taka.sum()}')

df.loc[df['Company'].str.upper() == 'COROLLA CROSS', ['Company','Model']] = ['TOYOTA','Corolla Cross']

df['Company'] = df['Company'].str.upper()
df

Shifted rows repaired: 23


,Company,Model,Year,Color,Engine_Capacity,Fuel_Type,Mileage,Auction_Grade,Price
0,TOYOTA,VOXY HYBRID,2020,BLACK,1800,HYBRID,69777,4.5,3500000
1,TOYOTA,ESQUIRE,2020,BLACK,1800,HYBRID,92500,3.5,3700000
2,TOYOTA,ESQUIRE,2020,PEARL,1800,HYBRID,97000,3.5,4000000
3,TOYOTA,COROLLA,2020,PEARL,1800,HYBRID,33000,4.5,3600000
4,TOYOTA,Corolla Cross,2021,Redwine,1800,Hybrid,30000,4.5,4850000
...,...,...,...,...,...,...,...,...,...
989,TOYOTA,AXIO HYBRID,2015,SILVER,NaN,NaN,52940,52940,1600000
990,TOYOTA,AXIO HYBRID,2015,BEIGE,1500,HYBRID,97000,4,1530000
991,TOYOTA,AXIO HYBRID,2015,SILVER,1500,Gasoline,74000,4,1650000
992,TOYOTA,CHR,2017,NaN,1800,Hybrid,1880,5,3150000


In [4]:
model_map = [
    ('axio',           'Axio'),
    ('axela',          'Axela'),
    ('fielder',        'Fielder'),
    ('corollacross',   'Corolla Cross'),
    ('corollatouring', 'Corolla Touring'),
    ('corolla',        'Corolla'),
    ('yariscross',     'Yaris Cross'),
    ('allion',         'Allion'),
    ('premio',         'Premio'),
    ('priusphv',       'Prius PHV'),
    ('priusalpha',     'Prius Alpha'),
    ('prius',          'Prius'),
    ('aqua',           'Aqua'),
    ('esquire',        'Esquire'),
    ('noah',           'Noah'),
    ('voxy',           'Voxy'),
    ('harrier',        'Harrier'),
    ('chr',            'CHR'),
    ('rav4',           'RAV4'),
    ('crown',          'Crown'),
    ('camry',          'Camry'),
    ('sienta',         'Sienta'),
    ('landcruiser',    'Land Cruiser Prado'),
    ('hiace',          'Hiace'),
    ('haice',          'Hiace'),
    ('alphard',        'Alphard'),
    ('vellfire',       'Vellfire'),
    ('vezel',          'Vezel'),
    ('crv',            'CR-V'),
    ('civic',          'Civic'),
    ('grace',          'Grace'),
    ('insight',        'Insight'),
    ('xtrail',         'X-Trail'),
    ('tekna',          'Tekna'),
    ('outlander',      'Outlander'),
    ('mazda3',         'Mazda 3'),
    ('cx3',            'CX-3'),
    ('carry',          'Carry'),
    ('nx300',          'NX300h'),
    ('es300',          'ES300h'),
    ('rx300',          'RX300'),
    ('gmodenero',      'Noah'),
]

def clean_model(x):
    if pd.isna(x): return x
    key = re.sub(r'[^a-z0-9]', '', x.lower())
    for k, std in model_map:
        if k in key:
            return std
    if re.match(r's($|\s)', x.lower()):
        return 'Prius'
    return x.title()

df['Model'] = df['Model'].map(clean_model)
df['Model'].value_counts()

Model
Prius                 165
Corolla Cross         163
Axio                  100
Premio                 64
CHR                    56
Fielder                42
Harrier                39
CR-V                   38
Corolla                37
Allion                 37
Esquire                31
Noah                   28
Grace                  22
X-Trail                19
Insight                18
Yaris Cross            18
Aqua                   16
Vezel                  15
Voxy                   15
Land Cruiser Prado     12
Hiace                  10
Civic                   9
RAV4                    6
Sienta                  4
Alphard                 4
Crown                   4
Outlander               3
Corolla Touring         3
NX300h                  3
Axela                   2
Mazda 3                 2
Carry                   1
Prius PHV               1
Camry                   1
ES300h                  1
Lexus                   1
RX300                   1
CX-3                    1
Tekna 

In [5]:
def clean_fuel(x):
    if pd.isna(x): return x
    s = x.lower()
    if 'non' in s and 'hybrid' in s:  return 'Petrol'
    if 'plug' in s:                   return 'Plug-in Hybrid'
    if re.search(r'h[yr]+b?r?[di]+d|hybrd|hyrbrid', s) or 'hybri' in s: return 'Hybrid'
    if s.startswith(('oct','pet','gas')): return 'Petrol'
    if s.startswith(('dis','des')):       return 'Diesel'
    return np.nan

df['Fuel_Type'] = df['Fuel_Type'].map(clean_fuel)
df['Fuel_Type'].value_counts()

Fuel_Type
Hybrid            689
Petrol            256
Diesel              3
Plug-in Hybrid      2
Name: count, dtype: int64

In [6]:
df['Color'] = df['Color'].str.title().replace({
    'Whte':    'White',
    'Grey':    'Gray',
    'Redwine': 'Wine Red',
    'Wine':    'Wine Red',
    'Twin':    np.nan,
})
df['Color'].value_counts

<bound method IndexOpsMixin.value_counts of 0         Black
1         Black
2         Pearl
3         Pearl
4      Wine Red
         ...   
989      Silver
990       Beige
991      Silver
992         NaN
993       Beige
Name: Color, Length: 994, dtype: object>

In [7]:
def to_num(s):
    sep = '.' if s.name == 'Auction_Grade' else ''
    return pd.to_numeric(s.str.replace(',', sep, regex=False), errors='coerce')

for c in ['Year', 'Engine_Capacity', 'Mileage', 'Price', 'Auction_Grade']:
    df[c] = to_num(df[c])

df.dtypes

Company             object
Model               object
Year               float64
Color               object
Engine_Capacity    float64
Fuel_Type           object
Mileage            float64
Auction_Grade      float64
Price              float64
dtype: object

In [8]:
df.loc[~df['Year'].between(2010, 2026), 'Year']                = np.nan
df.loc[df['Engine_Capacity'] > 10000,   'Engine_Capacity']    /= 10           # 18000 → 1800
df.loc[df['Engine_Capacity'] > 1000,    'Engine_Capacity']     = \
    (df['Engine_Capacity'] / 100).round() * 100                               # 1498 → 1500
df.loc[df['Engine_Capacity'] < 600,     'Engine_Capacity']     = np.nan
df.loc[df['Auction_Grade'] > 6,         'Auction_Grade']       = np.nan
df.loc[df['Mileage'] < 100,             'Mileage']             = np.nan       # 0, 11 = typos

# Price typo: value is 10× too high vs. model median (e.g. Axio 25,800,000 → 2,580,000)
med = df.groupby('Model')['Price'].transform('median')
big = df['Price'] > 6 * med
df.loc[big & ((df['Price'] / 10) / med).between(0.5, 1.8), 'Price'] /= 10

df.describe().T

,count,mean,std,min,25%,50%,75%,max
Year,986.0,2.019735e+03,1.753868e+00,2015.0,2019.0,2019.0,2021.0,2025.0
Engine_Capacity,982.0,1.734063e+03,2.402282e+02,650.0,1500.0,1800.0,1800.0,2700.0
Mileage,913.0,3.999593e+04,2.430364e+04,114.0,22593.0,35000.0,53000.0,145114.0
Auction_Grade,857.0,4.289965e+00,4.594462e-01,3.0,4.0,4.5,4.5,6.0
Price,958.0,4.260663e+06,2.002082e+06,300000.0,3200000.0,3750000.0,4850000.0,18000000.0


In [9]:
before = len(df)
df = df.dropna(subset=['Model', 'Year', 'Price']).drop_duplicates()
print(f'Dropped (missing key / duplicates): {before - len(df)} rows')
df

Dropped (missing key / duplicates): 66 rows


,Company,Model,Year,Color,Engine_Capacity,Fuel_Type,Mileage,Auction_Grade,Price
0,TOYOTA,Voxy,2020.0,Black,1800.0,Hybrid,69777.0,4.5,3500000.0
1,TOYOTA,Esquire,2020.0,Black,1800.0,Hybrid,92500.0,3.5,3700000.0
2,TOYOTA,Esquire,2020.0,Pearl,1800.0,Hybrid,97000.0,3.5,4000000.0
3,TOYOTA,Corolla,2020.0,Pearl,1800.0,Hybrid,33000.0,4.5,3600000.0
4,TOYOTA,Corolla Cross,2021.0,Wine Red,1800.0,Hybrid,30000.0,4.5,4850000.0
...,...,...,...,...,...,...,...,...,...
989,TOYOTA,Axio,2015.0,Silver,NaN,NaN,52940.0,NaN,1600000.0
990,TOYOTA,Axio,2015.0,Beige,1500.0,Hybrid,97000.0,4.0,1530000.0
991,TOYOTA,Axio,2015.0,Silver,1500.0,Petrol,74000.0,4.0,1650000.0
992,TOYOTA,CHR,2017.0,NaN,1800.0,Hybrid,1880.0,5.0,3150000.0


In [10]:
def iqr_ok(s, k=1.5):
    if s.count() < 8:
        return pd.Series(True, index=s.index)
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    return s.between(q1 - k * iqr, q3 + k * iqr) | s.isna()

keep_price   = df.groupby('Model')['Price'].transform(iqr_ok).astype(bool)
keep_mileage = df.groupby('Model')['Mileage'].transform(iqr_ok).astype(bool)

before = len(df)
df = df[keep_price & keep_mileage].reset_index(drop=True)
print(f'Outliers removed: {before - len(df)} rows')
df

Outliers removed: 28 rows


,Company,Model,Year,Color,Engine_Capacity,Fuel_Type,Mileage,Auction_Grade,Price
0,TOYOTA,Voxy,2020.0,Black,1800.0,Hybrid,69777.0,4.5,3500000.0
1,TOYOTA,Esquire,2020.0,Black,1800.0,Hybrid,92500.0,3.5,3700000.0
2,TOYOTA,Esquire,2020.0,Pearl,1800.0,Hybrid,97000.0,3.5,4000000.0
3,TOYOTA,Corolla,2020.0,Pearl,1800.0,Hybrid,33000.0,4.5,3600000.0
4,TOYOTA,Corolla Cross,2021.0,Wine Red,1800.0,Hybrid,30000.0,4.5,4850000.0
...,...,...,...,...,...,...,...,...,...
895,TOYOTA,CHR,2017.0,NaN,1800.0,Hybrid,28000.0,NaN,3250000.0
896,TOYOTA,Axio,2015.0,Silver,1500.0,Hybrid,31997.0,4.0,1660000.0
897,TOYOTA,Axio,2015.0,Silver,1500.0,Petrol,74000.0,4.0,1650000.0
898,TOYOTA,CHR,2017.0,NaN,1800.0,Hybrid,1880.0,5.0,3150000.0


In [11]:
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isna().sum())
print('\nPrice by model:')
df.groupby('Model')['Price'].agg(['count','min','median','max']).sort_values('count', ascending=False)

Shape: (900, 9)

Missing values:
Company              0
Model                0
Year                 0
Color              117
Engine_Capacity      5
Fuel_Type           20
Mileage             44
Auction_Grade       91
Price                0
dtype: int64

Price by model:


,count,min,median,max
Model,,,,
Prius,150,2500000.0,3300000.0,4100000.0
Corolla Cross,147,4200000.0,5100000.0,6200000.0
Axio,94,1650000.0,2600000.0,3500000.0
CHR,55,2950000.0,3500000.0,3850000.0
Premio,55,3100000.0,4350000.0,4900000.0
Fielder,38,2050000.0,2800000.0,3450000.0
CR-V,36,5450000.0,5900000.0,6800000.0
Corolla,35,2850000.0,3350000.0,3800000.0
Harrier,33,6350000.0,8200000.0,11200000.0


In [12]:
df = df.convert_dtypes()
df.to_csv('u_import.csv', index=False)
print('Saved u_import.csv')

Saved u_import.csv


In [13]:
df

,Company,Model,Year,Color,Engine_Capacity,Fuel_Type,Mileage,Auction_Grade,Price
0,TOYOTA,Voxy,2020,Black,1800,Hybrid,69777,4.5,3500000
1,TOYOTA,Esquire,2020,Black,1800,Hybrid,92500,3.5,3700000
2,TOYOTA,Esquire,2020,Pearl,1800,Hybrid,97000,3.5,4000000
3,TOYOTA,Corolla,2020,Pearl,1800,Hybrid,33000,4.5,3600000
4,TOYOTA,Corolla Cross,2021,Wine Red,1800,Hybrid,30000,4.5,4850000
...,...,...,...,...,...,...,...,...,...
895,TOYOTA,CHR,2017,<NA>,1800,Hybrid,28000,<NA>,3250000
896,TOYOTA,Axio,2015,Silver,1500,Hybrid,31997,4.0,1660000
897,TOYOTA,Axio,2015,Silver,1500,Petrol,74000,4.0,1650000
898,TOYOTA,CHR,2017,<NA>,1800,Hybrid,1880,5.0,3150000
